In [1]:
from evo.tools import file_interface
from evo.core import sync
from evo.tools import plot
import matplotlib.pyplot as plt
from evo.core import metrics
from rosbags.rosbag1 import Reader
import copy

In [2]:
with Reader('C:\\Users\\mauro\\Desktop\\Progretto SML\\ros-exploration-config\\output\\32D-8\\run1\\32D-8.bag') as reader:
    traj_est = file_interface.read_bag_trajectory(reader,'/odom')
    traj_ref = file_interface.read_bag_trajectory(reader,'/base_pose_ground_truth')
traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est)

In [3]:
traj_est_aligned = copy.deepcopy(traj_est)
traj_est_aligned.align(traj_ref, correct_scale=False, correct_only_scale=False)
traj_est_origin = copy.deepcopy(traj_est)
traj_est_origin.align_origin(traj_ref)

array([[  1.        ,   0.        ,   0.        ,  -0.56666667],
       [  0.        ,   1.        ,   0.        , -25.23333333],
       [  0.        ,   0.        ,   1.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   1.        ]])

In [4]:
ax = plt.gca()
traj_by_label = {
    "estimate": traj_est_origin,
    "reference": traj_ref
}
plot.trajectories(ax, traj_by_label, plot.PlotMode.xy)
ax.set_title('Robot trajectories')
ax.set_xlabel('x(m)')
ax.set_ylabel('y(m)')
ax.legend(fontsize=20)
ax.tick_params(axis='both', labelsize=24)
ax.xaxis.label.set_size(28)
ax.yaxis.label.set_size(28)
ax.title.set_fontsize(30)
plt.show()

In [5]:
ape_metric = metrics.APE(metrics.PoseRelation.translation_part)
ape_metric.process_data((traj_ref, traj_est_origin))

In [7]:
ape_stats = ape_metric.get_all_statistics()
seconds_from_start = [t - traj_est.timestamps[0] for t in traj_est.timestamps]
fig = plt.figure()
ax = fig.gca()
plot.error_array(ax, ape_metric.error, x_array=seconds_from_start,
                 statistics={s:v for s,v in ape_stats.items() if s != "sse"},
                 name="ATE", title="APE w.r.t. " + ape_metric.pose_relation.value, xlabel="$t$ (s)")
ax.set_title('ATE')
ax.title.set_fontsize(30)
ax.set_ylabel('ATE(m)')
ax.tick_params(axis='both', labelsize=24)
ax.xaxis.label.set_size(28)
ax.yaxis.label.set_size(28)
ax.legend(fontsize=20)
plt.show()
traj_est.timestamps[0]

np.float64(0.1)